# End-to-End QSR Analytics Walkthrough

This notebook demonstrates the project workflow using synthetic restaurant operations data only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent / 'src') if Path.cwd().name == 'notebooks' else str(Path.cwd() / 'src'))
from qsr_analytics.metrics import add_kpis
from qsr_analytics.did import manual_did, fit_twfe_did

## 1. Generate synthetic data
Run `python scripts/generate_synthetic_data.py` from the repository root before executing the remaining cells.

In [ ]:
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(root / 'data/generated/store_daily.csv', parse_dates=['date'])
df = add_kpis(df)
df.head()

## 2. KPI analysis
Store/day is the analytical grain. Do not join raw employee-shift rows directly to this table because doing so multiplies store sales across shifts.

In [ ]:
kpis = ['net_sales_gbp','transactions','avg_transaction_value_gbp','transactions_per_paid_hour','labour_cost_pct','waste_pct_sales','avg_service_seconds','complaints_per_1000_orders']
df.groupby('treated_store')[kpis].mean().round(2)

## 3. Difference-in-differences
The manual estimator is `(Treated Post - Treated Pre) - (Control Post - Control Pre)`.

In [ ]:
manual_did(df, 'transactions_per_paid_hour')

## 4. Two-way fixed-effects DID
The regression controls for time-invariant store differences and common date shocks. Standard errors are clustered by store.

In [ ]:
model = fit_twfe_did(df, 'transactions_per_paid_hour')
model.summary().tables[1]

## 5. Interpretation
A positive DID coefficient for transactions per paid hour means treated stores improved productivity more than the control stores over the same period. Because the dataset is synthetic, this is a demonstration result rather than a claim about any real business.